# WakeWord Workbench to microWakeWord: Complete Training Workflow

This notebook demonstrates the complete pipeline from dataset generation with WakeWord Workbench to training a production-ready wake word model using microWakeWord.

## What You Will Learn

1. **Environment Setup** - Install all required dependencies
2. **Dataset Generation** - Use WakeWord Workbench to create positive and negative samples
3. **Data Conversion** - Convert workbench output to microWakeWord's RaggedMmap format
4. **Feature Extraction** - Generate spectrogram features using microfrontend
5. **Training Configuration** - Set up YAML config for microWakeWord training
6. **Model Training** - Train a MixedNet architecture model
7. **Evaluation** - Test model performance with FAR/FRR metrics
8. **Export** - Generate TFLite model for deployment

## Prerequisites

- Python 3.10 or 3.11
- 4GB+ RAM (8GB recommended)
- ~2GB free disk space
- CPU-compatible (GPU optional but not required)

## Estimated Total Runtime

| Stage | CPU Runtime | GPU Runtime |
|-------|-------------|-------------|
| Setup & Installation | 5-10 min | 5-10 min |
| Dataset Generation | 10-20 min | 10-20 min |
| Feature Extraction | 5-10 min | 5-10 min |
| Training (quick test) | 15-30 min | 5-10 min |
| Training (production) | 2-4 hours | 30-60 min |
| Evaluation & Export | 5-10 min | 5-10 min |

**Note:** This notebook is designed to run on CPU. GPU acceleration will speed up training but is not required.

## Section 1: Environment Setup

**Estimated Runtime:** 5-10 minutes

Install all required packages. This includes:
- WakeWord Workbench for dataset generation
- microWakeWord for training
- Dependencies for audio processing and feature extraction

In [ ]:
# Check Python version
import sys
print(f"Python version: {sys.version}")

if sys.version_info < (3, 10) or sys.version_info >= (3, 12):
    print("WARNING: This notebook requires Python 3.10 or 3.11")
    print("Please switch to a compatible Python version")

In [ ]:
# Detect runtime environment and uv status
import os
import sys
import subprocess

def is_uv_venv():
    """Check if running inside a uv-managed virtual environment."""
    return (
        os.environ.get("VIRTUAL_ENV") is not None
        and os.environ.get("UV") is not None
    ) or (
        os.environ.get("VIRTUAL_ENV") is not None
        and "uv" in os.environ.get("PATH", "")
    )

def run_install(cmd, package_name):
    """Run install command, handling externally-managed-environment errors."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"  {package_name}: installed")
        return True
    # uv-managed env: fall back to pip with --break-system-packages
    if "externally-managed-environment" in result.stderr:
        alt_cmd = ["pip", "install", "--break-system-packages"] + cmd[2:]
        result = subprocess.run(alt_cmd, capture_output=True, text=True)
        if result.returncode == 0:
            print(f"  {package_name}: installed (--break-system-packages)")
            return True
    if "already satisfied" in result.stdout.lower() or "already satisfied" in result.stderr.lower():
        print(f"  {package_name}: already satisfied")
        return True
    print(f"  WARNING: {package_name} install issue ({result.returncode})")
    if result.stderr:
        for line in result.stderr.strip().splitlines()[:5]:
            print(f"    {line}")
    return False

RUNNING_IN_UV = is_uv_venv()
print(f"Environment: {'uv venv' if RUNNING_IN_UV else 'standard Python'}")
print(f"Python: {sys.version_info.major}.{sys.version_info.minor}")

In [ ]:
# Install WakeWord Workbench
# Idempotent: skips install if already importable.

import os
import sys

# 1. Check if already available
try:
    import wakeword_workbench
    print("WakeWord Workbench: already importable (skipping install)")
except ImportError:
    print("WakeWord Workbench: not found, attempting install...")

    if os.path.exists("pyproject.toml") and os.path.exists("src/wakeword_workbench"):
        print("Installing from local source...")
        result = os.system("pip install -e . > /dev/null 2>&1")
        if result == 0:
            print("  Installed from local source")
        elif os.environ.get("VIRTUAL_ENV"):
            result = os.system("pip install -e . --break-system-packages > /dev/null 2>&1")
            if result == 0:
                print("  Installed from local source (--break-system-packages)")
        else:
            print("  Install failed - try: uv sync")
    else:
        print("Installing from PyPI...")
        result = os.system("pip install wakeword-workbench > /dev/null 2>&1")
        if result == 0:
            print("  Installed from PyPI")
        elif os.environ.get("VIRTUAL_ENV"):
            result = os.system("pip install wakeword-workbench --break-system-packages > /dev/null 2>&1")
            if result == 0:
                print("  Installed from PyPI (--break-system-packages)")

    # Reload to pick up new install
    import importlib
    import site
    site.main()
    try:
        importlib.invalidate_caches()
        import wakeword_workbench  # noqa: F401
        print("WakeWord Workbench: now importable")
    except ImportError:
        print("WARNING: WakeWord Workbench still not importable after install")
        print("  Try: uv sync && uv run jupyter notebook")

In [ ]:
# Install microWakeWord dependencies: pymicro-features + audio-metadata
# Idempotent: skips if packages are already importable.

import platform
import os

system = platform.system()
print(f"Detected OS: {system}")

# Check what's already available
pymicro_available = False
try:
    import pymicro_features  # noqa: F401
    pymicro_available = True
    print("pymicro-features: already importable (skipping)")
except ImportError:
    pass

audiometadata_available = False
try:
    import audio_metadata  # noqa: F401
    audiometadata_available = True
    print("audio-metadata: already importable (skipping)")
except ImportError:
    pass

# Install pymicro-features if missing
if not pymicro_available:
    if system == "Darwin":
        cmd = "pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'"
    else:
        cmd = "pip install pymicro-features"
    print(f"Installing pymicro-features...")
    result = os.system(f"{cmd} > /dev/null 2>&1")
    if result == 0:
        print("  pymicro-features: installed")
    elif os.environ.get("VIRTUAL_ENV"):
        alt = cmd.replace("pip install", "pip install --break-system-packages")
        result = os.system(f"{alt} > /dev/null 2>&1")
        if result == 0:
            print("  pymicro-features: installed (--break-system-packages)")
        else:
            print("  pymicro-features: install failed")
    else:
        print("  pymicro-features: install failed")

# Install audio-metadata if missing
if not audiometadata_available:
    cmd = "pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'"
    print("Installing audio-metadata...")
    result = os.system(f"{cmd} > /dev/null 2>&1")
    if result == 0:
        print("  audio-metadata: installed")
    elif os.environ.get("VIRTUAL_ENV"):
        alt = cmd.replace("pip install", "pip install --break-system-packages")
        result = os.system(f"{alt} > /dev/null 2>&1")
        if result == 0:
            print("  audio-metadata: installed (--break-system-packages)")
        else:
            print("  audio-metadata: install failed")
    else:
        print("  audio-metadata: install failed")

In [ ]:
# Clone and install microWakeWord
# Idempotent: skips if directory exists and package is importable.

import os
import importlib
import site

# Check if already importable
try:
    import microwakeword  # noqa: F401
    print("microWakeWord: already importable (skipping)")
except ImportError:
    print("microWakeWord: not found, installing...")

    # Clone if directory missing
    if not os.path.exists("microWakeWord"):
        print("Cloning microWakeWord repository...")
        os.system("git clone https://github.com/kahrendt/microWakeWord > /dev/null 2>&1")
    else:
        print("microWakeWord directory already exists")

    # Install editable
    print("Installing microWakeWord...")
    result = os.system("pip install -e ./microWakeWord > /dev/null 2>&1")
    if result == 0:
        print("  microWakeWord: installed")
    elif os.environ.get("VIRTUAL_ENV"):
        result = os.system("pip install -e ./microWakeWord --break-system-packages > /dev/null 2>&1")
        if result == 0:
            print("  microWakeWord: installed (--break-system-packages)")
        else:
            print("  microWakeWord: install failed")
    else:
        print("  microWakeWord: install failed")

    # Reload site to pick up new install
    site.main()
    importlib.invalidate_caches()

    try:
        import microwakeword  # noqa: F401
        print("microWakeWord: now importable")
    except ImportError:
        print("WARNING: microWakeWord not importable - may need kernel restart")

In [ ]:
# Verify all installed packages
import os

RUNNING_IN_UV = os.environ.get("VIRTUAL_ENV") is not None
print(f"Environment: {'uv venv' if RUNNING_IN_UV else 'standard Python'}")
print("=" * 50)

status = {}

# Check each package
packages = {
    "WakeWord Workbench": "wakeword_workbench",
    "microWakeWord": "microwakeword",
    "mmap_ninja": "mmap_ninja",
    "TensorFlow": "tensorflow",
    "pymicro-features": "pymicro_features",
    "audio-metadata": "audio_metadata",
}

for name, module in packages.items():
    try:
        mod = __import__(module)
        ver = getattr(mod, "__version__", "installed")
        print(f"  {name}: OK ({ver})")
        status[name] = True
    except ImportError:
        print(f"  {name}: MISSING")
        status[name] = False

print("=" * 50)

# TensorFlow GPU info
if status.get("TensorFlow"):
    import tensorflow as tf
    gpus = tf.config.list_physical_devices("GPU")
    print(f"  GPU available: {len(gpus) > 0}")

# Summary
missing = [k for k, v in status.items() if not v]
if missing:
    print(f"\nMissing packages: {', '.join(missing)}")
    print("If using uv:  uv sync && uv run jupyter notebook")
    print("Or re-run Section 1 cells above")
else:
    print("\nAll packages verified! Environment is ready.")

## Section 2: Dataset Generation with WakeWord Workbench

**Estimated Runtime:** 10-20 minutes

Generate training data using WakeWord Workbench. This creates:
- Positive samples: TTS-generated wake word utterances
- Negative samples: Phonetic confusion phrases and synthetic negatives

**Important:** For microWakeWord, we disable workbench augmentation and export raw WAV files. microWakeWord has its own native augmentation pipeline.

In [ ]:
# Create configuration for WakeWord Workbench
# This config generates raw WAV files without augmentation

config_content = """
# WakeWord Workbench Configuration for microWakeWord Training
wake_word: \"hey vera\"

samples:
  positives: 500  # Start with 500 for quick testing (use 5000+ for production)
  negatives_multiplier: 5  # 2500 negative samples

tts:
  backend: \"kokoro\"
  voices:
    - \"af_sarah\"
    - \"am_adam\"
  speed: 1.0

augmentation:
  # DISABLE for microWakeWord - use native augmentation instead
  noise_snr: [0, 0]
  reverb_probability: 0.0
  gain_range: [0, 0]

output:
  path: \"./output/microwakeword\"
  format: []  # Export WAV only, no features
"""

# Write the config file
with open("workbench_config.yaml", "w") as f:
    f.write(config_content)

print("Configuration file created: workbench_config.yaml")
print("\nConfiguration preview:")
print(config_content)

In [ ]:
# Validate the configuration before running
from wakeword_workbench.config import load_config

try:
    config = load_config("workbench_config.yaml")
    print("Configuration is valid!")
    print(f"Wake word: {config.wake_word}")
    print(f"Positive samples: {config.samples.positives}")
    print(f"TTS backend: {config.tts.backend}")
except Exception as e:
    print(f"Configuration error: {e}")
    raise

In [ ]:
# Run WakeWord Workbench to generate the dataset
# This will create audio files in ./output/microwakeword/audio/

import subprocess

print("Starting dataset generation...")
print("This may take 10-20 minutes depending on your system.\n")

result = subprocess.run(
    ["python", "-m", "wakeword_workbench.cli", "run", "workbench_config.yaml"],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)

if result.returncode != 0:
    print("STDERR:")
    print(result.stderr)
    print(f"\nError: Process returned code {result.returncode}")
else:
    print("\nDataset generation completed successfully!")

In [ ]:
# Verify the generated dataset
from pathlib import Path
import json

output_dir = Path("./output/microwakeword")

# Check directory structure
print("Generated dataset structure:")
for item in sorted(output_dir.rglob("*")):
    if item.is_file():
        print(f"  {item.relative_to(output_dir)}")

# Load and inspect a manifest
manifest_path = output_dir / "train.jsonl"
if manifest_path.exists():
    with open(manifest_path) as f:
        lines = f.readlines()[:3]  # First 3 entries
    
    print(f"\nSample entries from train.jsonl:")
    for line in lines:
        entry = json.loads(line)
        print(f"  - {entry['path']} (label={entry['label']})")
    
    # Count entries
    with open(manifest_path) as f:
        count = sum(1 for _ in f)
    print(f"\nTotal training samples: {count}")

## Section 3: Download Augmentation Assets

**Estimated Runtime:** 5-10 minutes

microWakeWord uses external datasets for audio augmentation:
- **Impulse responses**: For room reverberation simulation
- **Background audio**: For noise injection (music, environmental sounds)

**License Note:** The data downloaded here has mixed licenses. Models trained with this data should be considered for non-commercial personal use only.

In [ ]:
# Download augmentation assets
# Adapted from microWakeWord basic_training_notebook

import os
import datasets
import scipy
import numpy as np
from pathlib import Path
from tqdm import tqdm

print("Downloading impulse responses (MIT RIRs)...")
output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    rir_dataset = datasets.load_dataset(
        "davidscripka/MIT_environmental_impulse_responses",
        split="train",
        streaming=True
    )
    # Save clips to 16-bit PCM wav files
    for row in tqdm(rir_dataset):
        name = row["audio"]["path"].split("/")[-1]
        scipy.io.wavfile.write(
            os.path.join(output_dir, name),
            16000,
            (row["audio"]["array"] * 32767).astype(np.int16)
        )
    print(f"Downloaded {len(list(Path(output_dir).glob('*.wav')))} impulse responses")
else:
    print("Impulse responses already downloaded")

print("\nDownloading background audio (AudioSet subset)...")
# Download a subset of AudioSet for background noise
audioset_dir = "./audioset_16k"
if not os.path.exists(audioset_dir):
    os.makedirs(audioset_dir, exist_ok=True)
    # Note: Downloading full AudioSet is slow; using a small subset
    print("Note: For production, download full AudioSet from HuggingFace")
    print("      datasets/agkphysics/AudioSet")
    print("\nSkipping full download for quick test. Using empty directory.")
else:
    print("Background audio directory ready")

## Section 4: Feature Extraction and RaggedMmap Conversion

**Estimated Runtime:** 5-10 minutes

This is the critical conversion step. WakeWord Workbench outputs WAV files and JSONL manifests, but microWakeWord expects:

1. **Spectrogram features**: 40-dim mel spectrograms using microfrontend
2. **RaggedMmap format**: Efficient memory-mapped storage

Key parameters:
- Sample rate: 16000 Hz
- Window size: 30 ms (480 samples)
- Window step: 10 ms (160 samples)
- Mel bands: 40
- Frequency range: 125-7500 Hz

In [ ]:
# Load audio clips from workbench output
from pathlib import Path
import numpy as np
import json

def load_workbench_manifest(manifest_path, audio_dir, max_samples=None):
    \"\"\"Load audio clips from workbench manifest.\"\"\"
    from scipy.io import wavfile
    
    clips = []
    labels = []
    
    with open(manifest_path) as f:
        lines = f.readlines()
    
    if max_samples:
        lines = lines[:max_samples]
    
    for line in lines:
        entry = json.loads(line)
        audio_path = audio_dir / entry["path"]
        
        try:
            sr, audio = wavfile.read(audio_path)
            
            # Convert to float32 and normalize
            if audio.dtype == np.int16:
                audio = audio.astype(np.float32) / 32767.0
            elif audio.dtype == np.int32:
                audio = audio.astype(np.float32) / 2147483647.0
            
            # Resample to 16kHz if needed
            if sr != 16000:
                print(f"Warning: {audio_path} has sample rate {sr}, expected 16000")
            
            clips.append(audio)
            labels.append(entry["label"])
        except Exception as e:
            print(f"Error loading {audio_path}: {e}")
    
    return clips, labels

# Load training data
audio_dir = Path("./output/microwakeword/audio")
manifest_path = Path("./output/microwakeword/train.jsonl")

print("Loading training clips...")
train_clips, train_labels = load_workbench_manifest(manifest_path, audio_dir)

print(f"Loaded {len(train_clips)} training clips")
print(f"Positive samples: {sum(train_labels)}")
print(f"Negative samples: {len(train_labels) - sum(train_labels)}")

In [ ]:
# Set up microWakeWord augmentation
# This is the native augmentation pipeline for microWakeWord

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips

# Create a simple clips wrapper for our loaded audio
class WorkbenchClips:
    \"\"\"Wrapper for workbench audio clips.\"\"\"
    
    def __init__(self, clips, labels):
        self.clips = clips
        self.labels = labels
    
    def get_random_clip(self, split="train"):
        idx = np.random.randint(len(self.clips))
        return self.clips[idx]
    
    def __len__(self):
        return len(self.clips)

clips = WorkbenchClips(train_clips, train_labels)

# Configure augmentation
# Adjust probabilities based on your dataset and requirements
augmenter = Augmentation(
    augmentation_duration_s=1.5,  # Match your clip duration
    augmentation_probabilities={
        "SevenBandParametricEQ": 0.1,
        "TanhDistortion": 0.1,
        "PitchShift": 0.1,
        "BandStopFilter": 0.1,
        "AddColorNoise": 0.1,
        "AddBackgroundNoise": 0.75,
        "Gain": 1.0,
        "RIR": 0.5,
    },
    impulse_paths=["mit_rirs"] if Path("mit_rirs").exists() else [],
    background_paths=["audioset_16k"] if Path("audioset_16k").exists() else [],
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.1,
    max_jitter_s=0.2,
)

print("Augmentation configured")
print(f"  Impulse responses: {augmenter.impulse_paths}")
print(f"  Background paths: {augmenter.background_paths}")

In [ ]:
# Generate spectrogram features
# This converts audio to microfrontend features and stores as RaggedMmap

from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap
import os

# Output directory for features
feature_dir = "./training_features"
os.makedirs(feature_dir, exist_ok=True)

# Create spectrogram generator
# step_ms=10: 10ms step for streaming compatibility
# slide_frames=10: Create multiple versions by sliding window
spectrograms = SpectrogramGeneration(
    clips=clips,
    augmenter=augmenter,
    step_ms=10,
    slide_frames=10,
)

print("Generating spectrogram features...")
print("This converts audio to 40-dim mel spectrograms\n")

# Generate for training set (with augmentation and sliding)
train_out = os.path.join(feature_dir, "training", "wakeword_mmap")
os.makedirs(train_out, exist_ok=True)

try:
    RaggedMmap.from_generator(
        out_dir=train_out,
        sample_generator=spectrograms.spectrogram_generator(split="train", repeat=2),
        batch_size=100,
        verbose=True,
    )
    print(f"\nTraining features saved to: {train_out}")
except Exception as e:
    print(f"Error generating training features: {e}")
    raise

In [ ]:
# Generate validation and testing features
# These use slide_frames=1 (no artificial sliding) for accurate evaluation

# Load validation data
val_manifest = Path("./output/microwakeword/val.jsonl")
if val_manifest.exists():
    print("Loading validation clips...")
    val_clips, val_labels = load_workbench_manifest(val_manifest, audio_dir)
    val_clip_obj = WorkbenchClips(val_clips, val_labels)
    
    # Validation spectrograms (no sliding)
    val_spectrograms = SpectrogramGeneration(
        clips=val_clip_obj,
        augmenter=augmenter,
        step_ms=10,
        slide_frames=1,  # No sliding for validation
    )
    
    val_out = os.path.join(feature_dir, "validation", "wakeword_mmap")
    os.makedirs(val_out, exist_ok=True)
    
    RaggedMmap.from_generator(
        out_dir=val_out,
        sample_generator=val_spectrograms.spectrogram_generator(split="validation", repeat=1),
        batch_size=100,
        verbose=True,
    )
    print(f"Validation features saved to: {val_out}")

# Load test data
test_manifest = Path("./output/microwakeword/test.jsonl")
if test_manifest.exists():
    print("\nLoading test clips...")
    test_clips, test_labels = load_workbench_manifest(test_manifest, audio_dir)
    test_clip_obj = WorkbenchClips(test_clips, test_labels)
    
    test_spectrograms = SpectrogramGeneration(
        clips=test_clip_obj,
        augmenter=augmenter,
        step_ms=10,
        slide_frames=1,
    )
    
    test_out = os.path.join(feature_dir, "testing", "wakeword_mmap")
    os.makedirs(test_out, exist_ok=True)
    
    RaggedMmap.from_generator(
        out_dir=test_out,
        sample_generator=test_spectrograms.spectrogram_generator(split="test", repeat=1),
        batch_size=100,
        verbose=True,
    )
    print(f"Test features saved to: {test_out}")

print("\nFeature extraction complete!")

In [ ]:
# Download pre-generated negative feature sets
# These are essential for training a robust model

import urllib.request
import zipfile

negative_dir = "./negative_datasets"
os.makedirs(negative_dir, exist_ok=True)

# List of negative datasets from HuggingFace
datasets_info = [
    ("dinner_party.zip", "Conversation/background speech"),
    ("dinner_party_eval.zip", "Evaluation dataset"),
    ("no_speech.zip", "Environmental sounds without speech"),
    ("speech.zip", "Speech samples"),
]

base_url = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"

for filename, description in datasets_info:
    zip_path = os.path.join(negative_dir, filename)
    extract_dir = os.path.join(negative_dir, filename.replace(".zip", ""))
    
    if os.path.exists(extract_dir):
        print(f"Skipping {filename} (already extracted)")
        continue
    
    print(f"Downloading {filename} ({description})...")
    try:
        urllib.request.urlretrieve(base_url + filename, zip_path)
        print(f"  Extracting...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(negative_dir)
        os.remove(zip_path)  # Clean up zip file
        print(f"  Done!")
    except Exception as e:
        print(f"  Error: {e}")
        print(f"  You may need to download manually from:")
        print(f"  {base_url}{filename}")

print("\nNegative datasets ready!")

## Section 5: Training Configuration

**Estimated Runtime:** 2 minutes

Create the YAML configuration file that controls the training process. Key parameters include:

- **Feature sets**: Define positive (wake word) and negative datasets
- **Sampling weights**: Control class balance during training
- **Training steps**: Number of optimization iterations
- **Class weights**: Penalize false positives more heavily
- **Model selection**: Two-stage metric-based weight selection

In [ ]:
# Create microWakeWord training configuration
# This is a quick-test configuration; adjust for production

import yaml

config = {
    # Training directory containing split subdirectories
    "train_dir": "./training_features",
    
    # Window step for spectrogram generation (must match feature extraction)
    "window_step_ms": 10,
    
    # Clip duration in milliseconds
    "clip_duration_ms": 1000,
    
    # Batch size (reduce if you run out of memory)
    "batch_size": 64,
    
    # Evaluation interval
    "eval_step_interval": 500,
}

# Define feature sets
# Each feature set includes:
# - features_dir: Path to RaggedMmap data
# - truth: True for wake word, False for negatives
# - sampling_weight: Frequency of selection (higher = more frequent)
# - penalty_weight: Loss multiplier for incorrect predictions
# - truncation_strategy: How to handle long clips
config["features"] = [
    {
        "features_dir": "./training_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "./negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "./negative_datasets/dinner_party",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "./negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "./negative_datasets/dinner_party_eval",
        "sampling_weight": 0.0,  # Only for validation/testing
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]

# Training schedule
# Lists are auto-padded to match len(training_steps)
config["training_steps"] = [3000]  # Quick test: 3000 steps
config["learning_rates"] = [0.001]

# Class weights (higher negative weight = more false positive rejection)
config["positive_class_weight"] = [1.0]
config["negative_class_weight"] = [10.0]

# Augmentation (applied during training)
config["mix_up_augmentation_prob"] = [0.0]
config["freq_mix_augmentation_prob"] = [0.0]
config["time_mask_max_size"] = [5]
config["time_mask_count"] = [2]
config["freq_mask_max_size"] = [5]
config["freq_mask_count"] = [2]

# Model selection metrics
# Two-stage selection: minimize primary, then maximize secondary
config["target_minimization"] = 0.05
config["maximization_metric"] = "val_recall"

# Write config file
with open("training_parameters.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("Training configuration created: training_parameters.yaml")
print("\nConfiguration preview:")
with open("training_parameters.yaml") as f:
    print(f.read())

## Section 6: Model Training

**Estimated Runtime:** 15-30 minutes (CPU) / 5-10 minutes (GPU)

Train the wake word model using microWakeWord's MixedNet architecture. The training process:

1. Loads features from RaggedMmap datasets
2. Trains a non-streaming model (for efficient batch training)
3. Evaluates on validation set periodically
4. Selects best weights using two-stage metric optimization
5. Converts to streaming TFLite format

**Architecture**: MixedNet uses depthwise separable convolutions with multiple kernel sizes for efficient multi-scale feature extraction.

In [ ]:
# Train the model
# This command runs the full training pipeline

import subprocess

print("Starting model training...")
print("This may take 15-30 minutes on CPU\n")
print("Note: Training will resume from checkpoint if interrupted\n")

# Training command with MixedNet architecture
cmd = [
    "python", "-m", "microwakeword.model_train_eval",
    "--training_config=training_parameters.yaml",
    "--train", "1",
    "--restore_checkpoint", "1",  # Resume if interrupted
    "--test_tf_nonstreaming", "0",
    "--test_tflite_nonstreaming", "0",
    "--test_tflite_nonstreaming_quantized", "0",
    "--test_tflite_streaming", "0",
    "--test_tflite_streaming_quantized", "1",  # Test streaming quantized model
    "--use_weights", "best_weights",
    "mixednet",
    "--pointwise_filters", "32,32,32,32",  # Quick-test size
    "--repeat_in_block", "1,1,1,1",
    "--mixconv_kernel_sizes", "[5], [9], [13], [21]",
    "--residual_connection", "0,0,0,0",
    "--first_conv_filters", "32",
    "--first_conv_kernel_size", "3",
    "--stride", "3",
]

print("Running command:")
print(" ".join(cmd))
print()

result = subprocess.run(cmd, capture_output=True, text=True)

print("STDOUT:")
print(result.stdout[-5000:] if len(result.stdout) > 5000 else result.stdout)  # Last 5000 chars

if result.returncode != 0:
    print("\nSTDERR:")
    print(result.stderr)
    print(f"\nTraining failed with return code: {result.returncode}")
else:
    print("\nTraining completed successfully!")

In [ ]:
# Check training output
from pathlib import Path

model_dir = Path("./trained_models/wakeword")

if model_dir.exists():
    print("Training output files:")
    for item in sorted(model_dir.rglob("*")):
        if item.is_file():
            size_mb = item.stat().st_size / (1024 * 1024)
            print(f"  {item.relative_to(model_dir)} ({size_mb:.2f} MB)")
    
    # Check for TFLite model
    tflite_path = model_dir / "tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
    if tflite_path.exists():
        size_kb = tflite_path.stat().st_size / 1024
        print(f"\nTFLite model size: {size_kb:.2f} KB")
        print("Model ready for deployment!")
else:
    print(f"Model directory not found: {model_dir}")
    print("Check training output above for errors.")

## Section 7: Training Visualization

**Estimated Runtime:** 1-2 minutes

Visualize training metrics including:
- Loss curves (training and validation)
- Accuracy over time
- False positive/negative rates
- Model selection points

In [ ]:
# Load training metrics
import json
from pathlib import Path

metrics_path = Path("./trained_models/wakeword/metrics.json")

if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    
    print(f"Available metrics: {list(metrics.keys())}")
    
    # Show sample of metrics
    for key, value in metrics.items():
        if isinstance(value, list) and len(value) > 0:
            print(f"\n{key}:")
            print(f"  Steps: {len(value)}")
            print(f"  First: {value[0]}")
            print(f"  Last: {value[-1]}")
else:
    print(f"Metrics file not found: {metrics_path}")
    print("Metrics may not have been saved or training is incomplete.")
    metrics = {}

In [ ]:
# Plot training metrics
import matplotlib.pyplot as plt
import numpy as np

# Set up the plotting style
plt.style.use('seaborn-v0_8-darkgrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

if metrics:
    # Plot 1: Loss curves
    ax = axes[0, 0]
    if 'loss' in metrics:
        ax.plot(metrics['loss'], label='Training Loss', alpha=0.8)
    if 'val_loss' in metrics:
        ax.plot(metrics['val_loss'], label='Validation Loss', alpha=0.8)
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('Training and Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Accuracy
    ax = axes[0, 1]
    if 'accuracy' in metrics:
        ax.plot(metrics['accuracy'], label='Training Accuracy', alpha=0.8)
    if 'val_accuracy' in metrics:
        ax.plot(metrics['val_accuracy'], label='Validation Accuracy', alpha=0.8)
    ax.set_xlabel('Step')
    ax.set_ylabel('Accuracy')
    ax.set_title('Training and Validation Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 3: False Positive Rate
    ax = axes[1, 0]
    if 'val_false_positive_rate' in metrics:
        ax.plot(metrics['val_false_positive_rate'], color='red', alpha=0.8)
        ax.set_xlabel('Step')
        ax.set_ylabel('False Positive Rate')
        ax.set_title('Validation False Positive Rate')
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'FPR data not available',
                ha='center', va='center', transform=ax.transAxes)
    
    # Plot 4: False Negative Rate
    ax = axes[1, 1]
    if 'val_false_negative_rate' in metrics:
        ax.plot(metrics['val_false_negative_rate'], color='orange', alpha=0.8)
        ax.set_xlabel('Step')
        ax.set_ylabel('False Negative Rate')
        ax.set_title('Validation False Negative Rate')
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'FNR data not available',
                ha='center', va='center', transform=ax.transAxes)
    
    plt.tight_layout()
    plt.savefig('training_metrics.png', dpi=150, bbox_inches='tight')
    print("Plot saved to: training_metrics.png")
else:
    print("No metrics available to plot")

plt.show()

## Section 8: Model Export and Deployment

**Estimated Runtime:** 2-3 minutes

The trained model is exported in TensorFlow Lite format for deployment. microWakeWord produces a streaming TFLite model that:

1. Accepts 40-dim spectrogram features as input
2. Maintains internal state for streaming inference
3. Outputs wake word probability (0-1)
4. Is quantized for efficient microcontroller deployment

**For ESPHome Integration:** You also need a model manifest JSON file. See the ESPHome documentation for details.

In [ ]:
# Export model to TFLite format
# The model should already be exported by the training script

from pathlib import Path
import shutil

# Source paths
model_dir = Path("./trained_models/wakeword")
tflite_source = model_dir / "tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"

# Destination
export_dir = Path("./exported_model")
export_dir.mkdir(exist_ok=True)

if tflite_source.exists():
    # Copy TFLite model
    tflite_dest = export_dir / "wakeword_model.tflite"
    shutil.copy(tflite_source, tflite_dest)
    print(f"TFLite model exported to: {tflite_dest}")
    
    # Get model size
    size_kb = tflite_dest.stat().st_size / 1024
    print(f"Model size: {size_kb:.2f} KB")
    
    # Copy training config for reference
    config_dest = export_dir / "training_parameters.yaml"
    shutil.copy("training_parameters.yaml", config_dest)
    print(f"Training config copied to: {config_dest}")
    
    # Create a README
    readme_content = f\"\"\"
# Exported Wake Word Model

## Model Information
- **Wake Word**: hey vera
- **Training Date**: {__import__('datetime').datetime.now().strftime('%Y-%m-%d')}
- **Model Size**: {size_kb:.2f} KB

## File Structure
- `wakeword_model.tflite` - Quantized streaming TFLite model
- `training_parameters.yaml` - Training configuration
- `training_metrics.png` - Training visualization (if available)

## Usage

### Python Inference
```python
import tensorflow as tf
import numpy as np

# Load model
interpreter = tf.lite.Interpreter(model_path="wakeword_model.tflite")
interpreter.allocate_tensors()

# Get input/output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Run inference (features shape: [batch, time_frames, 40])
interpreter.set_tensor(input_details[0]["index"], features)
interpreter.invoke()
probability = interpreter.get_tensor(output_details[0]["index"])
```

### ESPHome Integration
To use this model in ESPHome, you need:
1. Copy `wakeword_model.tflite` to your ESPHome configuration
2. Create a model manifest JSON file
3. Configure the `micro_wake_word` component

See: https://esphome.io/components/micro_wake_word
"""\n    
    readme_path = export_dir / "README.md"
    with open(readme_path, "w") as f:
        f.write(readme_content)
    print(f"README created: {readme_path}")
    
else:
    print(f"TFLite model not found: {tflite_source}")
    print("Training may have failed or not completed.")

In [ ]:
# Test the exported model with sample inference
import tensorflow as tf
import numpy as np

model_path = "./exported_model/wakeword_model.tflite"

if Path(model_path).exists():
    try:
        # Load TFLite model
        interpreter = tf.lite.Interpreter(model_path=model_path)
        interpreter.allocate_tensors()
        
        # Get model details
        input_details = interpreter.get_input_details()
        output_details = interpreter.get_output_details()
        
        print("Model loaded successfully!")
        print(f"\nInput shape: {input_details[0]["shape"]}")
        print(f"Input dtype: {input_details[0]["dtype"]}")
        print(f"Output shape: {output_details[0]["shape"]}")
        print(f"Output dtype: {output_details[0]["dtype"]}")
        
        # Test with dummy input
        # Shape: [batch=1, time_frames, 40 features]
        dummy_input = np.zeros((1, 100, 40), dtype=np.float32)
        interpreter.set_tensor(input_details[0]["index"], dummy_input)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]["index"])
        
        print(f"\nTest inference completed!")
        print(f"Output probability: {output[0][0]:.4f}")
        
    except Exception as e:
        print(f"Error loading/testing model: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"Model not found: {model_path}")

## Section 9: Next Steps and Production Tips

You have completed the end-to-end training pipeline! Here are recommendations for production deployment:

### 1. Increase Dataset Size
```python
# In workbench_config.yaml, increase:
samples:
  positives: 5000  # Was 500
  negatives_multiplier: 10  # More negative variety
```

### 2. Use More TTS Voices
```yaml
tts:
  voices:
    - af_sarah
    - am_adam
    - af_bella
    - af_nicole
    # Add more voices for diversity
```

### 3. Adjust Training Parameters
```yaml
# In training_parameters.yaml:
training_steps: [50000]  # More steps for better convergence
negative_class_weight: [20.0]  # Stronger false positive rejection
batch_size: 128  # Larger batch if you have GPU
```

### 4. Mine Hard Negatives
Use WakeWord Workbench's mining feature to extract false positives from real audio:
```bash
wakeword-workbench mine \
  --model ./exported_model/wakeword_model.tflite \
  --audio "recordings/*.wav" \
  --output ./mined_negatives
```

### 5. ESPHome Integration
Create a model manifest JSON file:
```json
{
  "model": "hey_vera",
  "trained_languages": ["en"],
  "parameters": {
    "probability_cutoff": 0.7,
    "sliding_window_size": 5,
    "tensor_arena_size": 10000
  }
}
```


## Summary

You have successfully completed the full wake word training pipeline:

1. Generated synthetic training data using WakeWord Workbench
2. Converted audio to microWakeWord's RaggedMmap format
3. Extracted 40-dim spectrogram features
4. Configured and trained a MixedNet model
5. Exported a quantized TFLite model
6. Visualized training metrics

### Key Files Generated

| File | Description |
|------|-------------|
| `./exported_model/wakeword_model.tflite` | Deployable model |
| `./training_parameters.yaml` | Training configuration |
| `./training_metrics.png` | Training visualization |
| `./trained_models/wakeword/` | Full training outputs |

### Troubleshooting

| Issue | Solution |
|-------|----------|
| High false positive rate | Increase `negative_class_weight`, add more negative samples |
| High false negative rate | Decrease `negative_class_weight`, add more positive samples |
| Out of memory | Reduce `batch_size`, reduce model size |
| Training too slow | Use GPU, reduce `training_steps` for testing |

### Resources

- [microWakeWord Repository](https://github.com/kahrendt/microWakeWord)
- [ESPHome microWakeWord Docs](https://esphome.io/components/micro_wake_word)
- [WakeWord Workbench Documentation](https://github.com/your-repo/wakeword-workbench)